In [111]:
import sqlite3
import pandas as pd

# Create a NEW database for DL prospects
conn = sqlite3.connect('nfl_analytics_dl.db')  # different name!
print("Connected to DL database")

# Your DL files
files = {
    'college_sacks': '../data/CollegeSacks13to25.csv',
    'college_tfls': '../data/CollegeTacklesForLoss13to25.csv',
    'college_tackles': '../data/CollegeTotalTackles13to25.csv',
    'combine': '../data/FullCombine13to26.csv',
    'nfl_stats': '../data/NFLStats18to25.csv',
    'tfl_ff': '../data/TFLsFFs18to25.csv'
}

# Load them the same way
for table_name, file_path in files.items():
    try:
        df = pd.read_csv(file_path)
        df.to_sql(table_name, conn, if_exists='replace', index=False)
        print(f"Loaded {table_name}: {len(df)} rows")
    except Exception as e:
        print(f"Error loading {file_path}: {e}")

conn.close()

Connected to DL database
Loaded college_sacks: 1652 rows
Loaded college_tfls: 1300 rows
Loaded college_tackles: 5453 rows
Loaded combine: 4730 rows
Loaded nfl_stats: 4000 rows
Loaded tfl_ff: 4100 rows


In [11]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('nfl_analytics_dl.db')

query = """
SELECT 
    n.Player as Name,
    c.Year as draft_year,
    c.Round as draft_round,
    c.PickOverall as overall_pick,

    -- Position classification
    CASE 
        -- Edge rushers
        WHEN n.Pos IN ('DE', 'LDE', 'RDE', 'DE/RDE', 'OLB', 'LOLB', 'ROLB', 'ROLB/LOLB', 'LOLB/ROLB', 'EDGE') THEN 'Edge'
        -- Interior DL
        WHEN n.Pos IN ('DT', 'NT', 'DL', 'DT/NG', 'DT/DE') THEN 'Interior'
        -- Catch any that don't fit (should be few)
        ELSE 'Other'
    END as position_group,

    
    -- Games and starts
    SUM(n.G) as total_games,
    SUM(n.GS) as games_started,
    
    -- Per game stats (pass rush)
    ROUND(CAST(SUM(n.Sk) AS FLOAT) / NULLIF(SUM(n.G), 0), 3) as sacks_per_game,
    ROUND(CAST(SUM(n.Prss) AS FLOAT) / NULLIF(SUM(n.G), 0), 3) as pressures_per_game,
    ROUND(CAST(SUM(n.QBKD) AS FLOAT) / NULLIF(SUM(n.G), 0), 3) as QBKDs_per_game,
    
    -- Per game stats (run defense)
    ROUND(CAST(SUM(t.TFL) AS FLOAT) / NULLIF(SUM(n.G), 0), 3) as tfl_per_game,
    ROUND(CAST(SUM(t.FF) AS FLOAT) / NULLIF(SUM(n.G), 0), 3) as ff_per_game,
    
    -- Career high stats (pass rush)
    MAX(n.Sk) as nfl_sacks_career_high,
    MAX(n.Prss) as nfl_pressures_career_high,
    MAX(n.QBKD) as nfl_QBKDs_career_high,
    MAX(n.Bats) as nfl_bats_career_high,
    MAX(n.Hrry) as nfl_hurries_career_high,
    
    -- Career high stats (run defense)
    MAX(t.TFL) as career_high_tfl,
    MAX(t.FF) as career_high_ff,
    
    -- Career totals
    SUM(n.Sk) as career_sacks,
    SUM(n.Prss) as career_pressures,
    SUM(t.TFL) as career_tfl,
    SUM(t.FF) as career_ff,
    
    -- Tackles (for context)
    SUM(t.SOLO) as career_solo_tackles,
    SUM(t.AST) as career_assist_tackles,
    ROUND(CAST(SUM(t.SOLO) AS FLOAT) / NULLIF(SUM(n.G), 0), 3) as solo_tackles_per_game

FROM nfl_stats n

-- Join combine data (draft info)
LEFT JOIN combine c ON n.Player = c.Player AND c.Pos IN ('DE', 'DT', 'DL', 'EDGE', 'OLB')  -- Exclude CB, S, WR, etc.

-- Join TFL/FF data
LEFT JOIN tfl_ff t ON n.Player = t.Name AND n.Year = t.Year

WHERE 
    n.Pos IN ('DE', 'DT', 'OLB', 'DL', 'LDE', 'RDE', 'LOLB', 'ROLB', 'DE/RDE', 'NT', 'DT/NG')
    AND n.G > 0  -- Players who actually played
    AND (n.Sk > 0 OR t.TFL > 0 OR t.FF > 0)  -- At least some production

GROUP BY 
    n.Player, c.Year, c.Round, c.PickOverall

ORDER BY 
    nfl_sacks_career_high DESC, 
    sacks_per_game DESC, 
    career_high_tfl DESC
"""
# LIMIT 50;

DL_summary = pd.read_sql_query(query, conn)

DL_summary.to_csv('dl_analysis_data.csv', index=False)

# print("TOP NFL PASS RUSHERS BY SACKS (with pressures as tiebreaker)")
# print("=" * 80)
# print(results.to_string(index=False))



conn.close()

# SUMMARY STATS

In [18]:
# Summary Stats
print("\n" + "=" * 80)
print("SUMMARY STATISTICS")

# percentiles = [25, 50, 75, 90, 95, 99]
percentiles = [25, 50, 75, 90, 99]

# PER GAME STATS
print("\nPER GAME STATS")

print("\nSACKS PER GAME PERCENTILES:")
print("-" * 40)
values = [DL_summary['sacks_per_game'].quantile(p/100) for p in percentiles]
for i, (p, value) in enumerate(zip(percentiles, values)):
    if i == 0:
        print(f"{p}th percentile: {value:.3f} sacks")
    else:
        diff = value - values[i-1]
        print(f"{p}th percentile: {value:.3f} sacks (+{diff:.3f})")


print("\nPRESSURES PER GAME PERCENTILES:")
print("-" * 40)
values = [DL_summary['pressures_per_game'].quantile(p/100) for p in percentiles]
for i, (p, value) in enumerate(zip(percentiles, values)):
    if i == 0:
        print(f"{p}th percentile: {value:.3f} pressures")
    else:
        diff = value - values[i-1]
        print(f"{p}th percentile: {value:.3f} pressures (+{diff:.3f})")


print("\nQBKDs PER GAME PERCENTILES:")
print("-" * 40)
values = [DL_summary['QBKDs_per_game'].quantile(p/100) for p in percentiles]
for i, (p, value) in enumerate(zip(percentiles, values)):
    if i == 0:
        print(f"{p}th percentile: {value:.3f} QBKDs")
    else:
        diff = value - values[i-1]
        print(f"{p}th percentile: {value:.3f} QBKDs (+{diff:.3f})")


print("\nTFLs PER GAME PERCENTILES:")
print("-" * 40)
values = [DL_summary['tfl_per_game'].quantile(p/100) for p in percentiles]
for i, (p, value) in enumerate(zip(percentiles, values)):
    if i == 0:
        print(f"{p}th percentile: {value:.3f} TFLs")
    else:
        diff = value - values[i-1]
        print(f"{p}th percentile: {value:.3f} TFLs (+{diff:.3f})")
        

print("\nFFs PER GAME PERCENTILES:")
print("-" * 40)
values = [DL_summary['ff_per_game'].quantile(p/100) for p in percentiles]
for i, (p, value) in enumerate(zip(percentiles, values)):
    if i == 0:
        print(f"{p}th percentile: {value:.3f} FFs")
    else:
        diff = value - values[i-1]
        print(f"{p}th percentile: {value:.3f} FFs (+{diff:.3f})")


# CAREER HIGH STATS
print("\nCAREER HIGH STATS")
print("\nCAREER HIGH SACKS PERCENTILES:")
print("-" * 40)
values = [DL_summary['nfl_sacks_career_high'].quantile(p/100) for p in percentiles]
for i, (p, value) in enumerate(zip(percentiles, values)):
    if i == 0:
        print(f"{p}th percentile: {value:.3f} sacks")
    else:
        diff = value - values[i-1]
        print(f"{p}th percentile: {value:.3f} sacks (+{diff:.3f})")


print("\nCAREER HIGH PRESSURE PERCENTILES:")
print("-" * 40)
values = [DL_summary['nfl_pressures_career_high'].quantile(p/100) for p in percentiles]
for i, (p, value) in enumerate(zip(percentiles, values)):
    if i == 0:
        print(f"{p}th percentile: {value:.3f} pressures")
    else:
        diff = value - values[i-1]
        print(f"{p}th percentile: {value:.3f} pressures (+{diff:.3f})")


print("\nCAREER HIGH QB KNOCKDOWN PERCENTILES:")
print("-" * 40)
values = [DL_summary['nfl_QBKDs_career_high'].quantile(p/100) for p in percentiles]
for i, (p, value) in enumerate(zip(percentiles, values)):
    if i == 0:
        print(f"{p}th percentile: {value:.3f} QBKDs")
    else:
        diff = value - values[i-1]
        print(f"{p}th percentile: {value:.3f} QBKDs (+{diff:.3f})")

print("\nCAREER HIGH TFL PERCENTILES:")
print("-" * 40)
values = [DL_summary['career_high_tfl'].quantile(p/100) for p in percentiles]
for i, (p, value) in enumerate(zip(percentiles, values)):
    if i == 0:
        print(f"{p}th percentile: {value:.3f} TFLs")
    else:
        diff = value - values[i-1]
        print(f"{p}th percentile: {value:.3f} TFLs (+{diff:.3f})")


print("\nCAREER HIGH FF PERCENTILES:")
print("-" * 40)
values = [DL_summary['career_high_ff'].quantile(p/100) for p in percentiles]
for i, (p, value) in enumerate(zip(percentiles, values)):
    if i == 0:
        print(f"{p}th percentile: {value:.3f} FFs")
    else:
        diff = value - values[i-1]
        print(f"{p}th percentile: {value:.3f} FFs (+{diff:.3f})")


###################################

# print("\nPLAYERS WITH 11 OR MORE SACKS (CAREER HIGH SEASON):")
# print("-" * 40)
elite_sacks = DL_summary[DL_summary['nfl_sacks_career_high'] >= 11]
# print(elite_sacks[['Name', 'nfl_sacks_career_high', 'nfl_pressures_career_high', 'career_nfl_sacks']].to_string(index=False))
#print(f"\nTotal players with 11+ sacks: {len(elite_sacks)}")


# print("\nPLAYERS WITH 36 OR MORE PRESSURES (CAREER HIGH SEASON):")
# print("-" * 40)
elite_pressures = DL_summary[DL_summary['nfl_pressures_career_high'] >= 36]
# print(elite_pressures[['Name', 'nfl_sacks_career_high', 'nfl_pressures_career_high', 'career_nfl_sacks']].to_string(index=False))
#print(f"\nTotal players with 36+ pressures: {len(elite_pressures)}")

# print("\nPLAYERS WITH 36 OR MORE PRESSURES (CAREER HIGH SEASON):")
# print("-" * 40)
elite_QBKDs = DL_summary[DL_summary['nfl_QBKDs_career_high'] >= 14]
# print(elite_QBKDs[['Name', 'nfl_sacks_career_high', 'nfl_pressures_career_high', 'nfl_QBKDs_career_high']].to_string(index=False))
#print(f"\nTotal players with 14+ QBKDs: {len(elite_QBKDs)}")



SUMMARY STATISTICS

PER GAME STATS

SACKS PER GAME PERCENTILES:
----------------------------------------
25th percentile: 0.094 sacks
50th percentile: 0.179 sacks (+0.085)
75th percentile: 0.324 sacks (+0.144)
90th percentile: 0.450 sacks (+0.126)
99th percentile: 0.820 sacks (+0.370)

PRESSURES PER GAME PERCENTILES:
----------------------------------------
25th percentile: 0.412 pressures
50th percentile: 0.705 pressures (+0.293)
75th percentile: 1.164 pressures (+0.459)
90th percentile: 1.673 pressures (+0.510)
99th percentile: 2.596 pressures (+0.922)

QBKDs PER GAME PERCENTILES:
----------------------------------------
25th percentile: 0.126 QBKDs
50th percentile: 0.235 QBKDs (+0.109)
75th percentile: 0.400 QBKDs (+0.165)
90th percentile: 0.564 QBKDs (+0.164)
99th percentile: 1.000 QBKDs (+0.436)

TFLs PER GAME PERCENTILES:
----------------------------------------
25th percentile: 0.225 TFLs
50th percentile: 0.333 TFLs (+0.109)
75th percentile: 0.500 TFLs (+0.167)
90th percentile:

***
### Summary Stats Takeaways

**1. The gap between 75th and 90th percentile is greater than the gap between 50th and 75th percentile producers.** Almost across the board, we see the per game and career high discrepencies greater among the 90th percentile players, with the exception being sacks per game and QBKDs per game. I think we contribute the closer discrepency in those areas among 90th percentile players to sacks being more of a volitile statistic based more on situation than any of the other statistics. It is hard to get to the quarterback and a player getting a hand on the QB is not always necessary to affect the game.

**2. 50th and 25th percentile players have very small margins.** The difference between a 50th and 25th percentile player seems to be very small, almost across the board. There doesn't appear to be a closer gap in production than these two groups. This means that a huge chunk of players fall into the slightly below average tier, at least when it comes to production. 

**The bottom line:** 

***

## Summary Stats for EDGE ONLY

In [20]:
# Summary Stats
print("\n" + "=" * 80)
print("EDGE SUMMARY STATISTICS")
edge_rushers = DL_summary[DL_summary['position_group'] == 'Edge']


# percentiles = [25, 50, 75, 90, 95, 99]
percentiles = [25, 50, 75, 90, 99]

# PER GAME STATS
print("\nPER GAME STATS")
print("\nSACKS PER GAME PERCENTILES:")
print("-" * 40)
values = [edge_rushers['sacks_per_game'].quantile(p/100) for p in percentiles]
for i, (p, value) in enumerate(zip(percentiles, values)):
    if i == 0:
        print(f"{p}th percentile: {value:.3f} sacks")
    else:
        diff = value - values[i-1]
        print(f"{p}th percentile: {value:.3f} sacks (+{diff:.3f})")

print("\nPRESSURES PER GAME PERCENTILES:")
print("-" * 40)
values = [edge_rushers['pressures_per_game'].quantile(p/100) for p in percentiles]
for i, (p, value) in enumerate(zip(percentiles, values)):
    if i == 0:
        print(f"{p}th percentile: {value:.3f} pressures")
    else:
        diff = value - values[i-1]
        print(f"{p}th percentile: {value:.3f} pressures (+{diff:.3f})")


print("\nQBKDs PER GAME PERCENTILES:")
print("-" * 40)
values = [edge_rushers['QBKDs_per_game'].quantile(p/100) for p in percentiles]
for i, (p, value) in enumerate(zip(percentiles, values)):
    if i == 0:
        print(f"{p}th percentile: {value:.3f} QBKDs")
    else:
        diff = value - values[i-1]
        print(f"{p}th percentile: {value:.3f} QBKDs (+{diff:.3f})")


print("\nTFLs PER GAME PERCENTILES:")
print("-" * 40)
values = [edge_rushers['tfl_per_game'].quantile(p/100) for p in percentiles]
for i, (p, value) in enumerate(zip(percentiles, values)):
    if i == 0:
        print(f"{p}th percentile: {value:.3f} TFLs")
    else:
        diff = value - values[i-1]
        print(f"{p}th percentile: {value:.3f} TFLs (+{diff:.3f})")


print("\nFFs PER GAME PERCENTILES:")
print("-" * 40)
values = [edge_rushers['ff_per_game'].quantile(p/100) for p in percentiles]
for i, (p, value) in enumerate(zip(percentiles, values)):
    if i == 0:
        print(f"{p}th percentile: {value:.3f} FFs")
    else:
        diff = value - values[i-1]
        print(f"{p}th percentile: {value:.3f} FFs (+{diff:.3f})")



# CAREER HIGH STATS
print("\nCAREER HIGH STATS")
print("\nCAREER HIGH SACKS PERCENTILES:")
print("-" * 40)
values = [edge_rushers['nfl_sacks_career_high'].quantile(p/100) for p in percentiles]
for i, (p, value) in enumerate(zip(percentiles, values)):
    if i == 0:
        print(f"{p}th percentile: {value:.3f} sacks")
    else:
        diff = value - values[i-1]
        print(f"{p}th percentile: {value:.3f} sacks (+{diff:.3f})")

print("\nCAREER HIGH PRESSURE PERCENTILES:")
print("-" * 40)
values = [edge_rushers['nfl_pressures_career_high'].quantile(p/100) for p in percentiles]
for i, (p, value) in enumerate(zip(percentiles, values)):
    if i == 0:
        print(f"{p}th percentile: {value:.3f} pressures")
    else:
        diff = value - values[i-1]
        print(f"{p}th percentile: {value:.3f} pressures (+{diff:.3f})")


print("\nCAREER HIGH QB KNOCKDOWN PERCENTILES:")
print("-" * 40)
values = [edge_rushers['nfl_QBKDs_career_high'].quantile(p/100) for p in percentiles]
for i, (p, value) in enumerate(zip(percentiles, values)):
    if i == 0:
        print(f"{p}th percentile: {value:.3f} QBKDs")
    else:
        diff = value - values[i-1]
        print(f"{p}th percentile: {value:.3f} QBKDs (+{diff:.3f})")

print("\nCAREER HIGH TFL PERCENTILES:")
print("-" * 40)
values = [edge_rushers['career_high_tfl'].quantile(p/100) for p in percentiles]
for i, (p, value) in enumerate(zip(percentiles, values)):
    if i == 0:
        print(f"{p}th percentile: {value:.3f} TFLs")
    else:
        diff = value - values[i-1]
        print(f"{p}th percentile: {value:.3f} TFLs (+{diff:.3f})")


print("\nCAREER HIGH FF PERCENTILES:")
print("-" * 40)
values = [edge_rushers['career_high_ff'].quantile(p/100) for p in percentiles]
for i, (p, value) in enumerate(zip(percentiles, values)):
    if i == 0:
        print(f"{p}th percentile: {value:.3f} FFs")
    else:
        diff = value - values[i-1]
        print(f"{p}th percentile: {value:.3f} FFs (+{diff:.3f})")


###################################

# print("\nPLAYERS WITH 11 OR MORE SACKS (CAREER HIGH SEASON):")
# print("-" * 40)
elite_sacks = edge_rushers[edge_rushers['nfl_sacks_career_high'] >= 11]
# print(elite_sacks[['Name', 'nfl_sacks_career_high', 'nfl_pressures_career_high', 'career_nfl_sacks']].to_string(index=False))
#print(f"\nTotal players with 11+ sacks: {len(elite_sacks)}")


# print("\nPLAYERS WITH 36 OR MORE PRESSURES (CAREER HIGH SEASON):")
# print("-" * 40)
elite_pressures = edge_rushers[edge_rushers['nfl_pressures_career_high'] >= 36]
# print(elite_pressures[['Name', 'nfl_sacks_career_high', 'nfl_pressures_career_high', 'career_nfl_sacks']].to_string(index=False))
#print(f"\nTotal players with 36+ pressures: {len(elite_pressures)}")

# print("\nPLAYERS WITH 36 OR MORE PRESSURES (CAREER HIGH SEASON):")
# print("-" * 40)
elite_QBKDs = edge_rushers[edge_rushers['nfl_QBKDs_career_high'] >= 14]
# print(elite_QBKDs[['Name', 'nfl_sacks_career_high', 'nfl_pressures_career_high', 'nfl_QBKDs_career_high']].to_string(index=False))
#print(f"\nTotal players with 14+ QBKDs: {len(elite_QBKDs)}")



EDGE SUMMARY STATISTICS

PER GAME STATS

SACKS PER GAME PERCENTILES:
----------------------------------------
25th percentile: 0.124 sacks
50th percentile: 0.231 sacks (+0.107)
75th percentile: 0.367 sacks (+0.136)
90th percentile: 0.505 sacks (+0.138)
99th percentile: 0.876 sacks (+0.370)

PRESSURES PER GAME PERCENTILES:
----------------------------------------
25th percentile: 0.585 pressures
50th percentile: 0.876 pressures (+0.291)
75th percentile: 1.378 pressures (+0.502)
90th percentile: 1.905 pressures (+0.527)
99th percentile: 2.837 pressures (+0.932)

QBKDs PER GAME PERCENTILES:
----------------------------------------
25th percentile: 0.156 QBKDs
50th percentile: 0.291 QBKDs (+0.135)
75th percentile: 0.445 QBKDs (+0.153)
90th percentile: 0.614 QBKDs (+0.170)
99th percentile: 1.022 QBKDs (+0.408)

TFLs PER GAME PERCENTILES:
----------------------------------------
25th percentile: 0.256 TFLs
50th percentile: 0.375 TFLs (+0.119)
75th percentile: 0.548 TFLs (+0.173)
90th percen

***
### EDGE Summary Stats Takeaways

**1. Pressures per game takes a HUGE jump after 50th percentile.** We can start to see what separates the average edge rushers from the good/great ones, at least in terms of getting after the QB. While the sacks in each percentile jump a similar amount, the pressure jump from the 50th to the 75th percentile then from the 75th to the 90th are both massive, at almost half a pressure per game. That far eclipses the sub .3 jump from the 25th to the 50th percentile. This again proves the earlier point that actually getting to the quarterback is hard to do, so the separation in stats where players actually get to the quarterback are hard to differentiate. But in terms of pressures, the great players do it at a much better clip than the average ones. 

**2. ** 

**The bottom line:** 

***

## Summary Stats for INTERIOR ONLY

In [22]:
# Summary Stats
print("\n" + "=" * 80)
print("INTERIOR SUMMARY STATISTICS")
interior_dl = DL_summary[DL_summary['position_group'] == 'Interior']


# percentiles = [25, 50, 75, 90, 95, 99]
percentiles = [25, 50, 75, 90, 99]

# PER GAME STATS
print("\nPER GAME STATS")
print("\nSACKS PER GAME PERCENTILES:")
print("-" * 40)
values = [interior_dl['sacks_per_game'].quantile(p/100) for p in percentiles]
for i, (p, value) in enumerate(zip(percentiles, values)):
    if i == 0:
        print(f"{p}th percentile: {value:.3f} sacks")
    else:
        diff = value - values[i-1]
        print(f"{p}th percentile: {value:.3f} sacks (+{diff:.3f})")

print("\nPRESSURES PER GAME PERCENTILES:")
print("-" * 40)
values = [interior_dl['pressures_per_game'].quantile(p/100) for p in percentiles]
for i, (p, value) in enumerate(zip(percentiles, values)):
    if i == 0:
        print(f"{p}th percentile: {value:.3f} pressures")
    else:
        diff = value - values[i-1]
        print(f"{p}th percentile: {value:.3f} pressures (+{diff:.3f})")


print("\nQBKDs PER GAME PERCENTILES:")
print("-" * 40)
values = [interior_dl['QBKDs_per_game'].quantile(p/100) for p in percentiles]
for i, (p, value) in enumerate(zip(percentiles, values)):
    if i == 0:
        print(f"{p}th percentile: {value:.3f} QBKDs")
    else:
        diff = value - values[i-1]
        print(f"{p}th percentile: {value:.3f} QBKDs (+{diff:.3f})")


print("\nTFLs PER GAME PERCENTILES:")
print("-" * 40)
values = [interior_dl['tfl_per_game'].quantile(p/100) for p in percentiles]
for i, (p, value) in enumerate(zip(percentiles, values)):
    if i == 0:
        print(f"{p}th percentile: {value:.3f} TFLs")
    else:
        diff = value - values[i-1]
        print(f"{p}th percentile: {value:.3f} TFLs (+{diff:.3f})")


print("\nFFs PER GAME PERCENTILES:")
print("-" * 40)
values = [interior_dl['ff_per_game'].quantile(p/100) for p in percentiles]
for i, (p, value) in enumerate(zip(percentiles, values)):
    if i == 0:
        print(f"{p}th percentile: {value:.3f} FFs")
    else:
        diff = value - values[i-1]
        print(f"{p}th percentile: {value:.3f} FFs (+{diff:.3f})")



# CAREER HIGH STATS
print("\nCAREER HIGH STATS")
print("\nCAREER HIGH SACKS PERCENTILES:")
print("-" * 40)
values = [interior_dl['nfl_sacks_career_high'].quantile(p/100) for p in percentiles]
for i, (p, value) in enumerate(zip(percentiles, values)):
    if i == 0:
        print(f"{p}th percentile: {value:.3f} sacks")
    else:
        diff = value - values[i-1]
        print(f"{p}th percentile: {value:.3f} sacks (+{diff:.3f})")

print("\nCAREER HIGH PRESSURE PERCENTILES:")
print("-" * 40)
values = [interior_dl['nfl_pressures_career_high'].quantile(p/100) for p in percentiles]
for i, (p, value) in enumerate(zip(percentiles, values)):
    if i == 0:
        print(f"{p}th percentile: {value:.3f} pressures")
    else:
        diff = value - values[i-1]
        print(f"{p}th percentile: {value:.3f} pressures (+{diff:.3f})")


print("\nCAREER HIGH QB KNOCKDOWN PERCENTILES:")
print("-" * 40)
values = [interior_dl['nfl_QBKDs_career_high'].quantile(p/100) for p in percentiles]
for i, (p, value) in enumerate(zip(percentiles, values)):
    if i == 0:
        print(f"{p}th percentile: {value:.3f} QBKDs")
    else:
        diff = value - values[i-1]
        print(f"{p}th percentile: {value:.3f} QBKDs (+{diff:.3f})")

print("\nCAREER HIGH TFL PERCENTILES:")
print("-" * 40)
values = [interior_dl['career_high_tfl'].quantile(p/100) for p in percentiles]
for i, (p, value) in enumerate(zip(percentiles, values)):
    if i == 0:
        print(f"{p}th percentile: {value:.3f} TFLs")
    else:
        diff = value - values[i-1]
        print(f"{p}th percentile: {value:.3f} TFLs (+{diff:.3f})")


print("\nCAREER HIGH FF PERCENTILES:")
print("-" * 40)
values = [interior_dl['career_high_ff'].quantile(p/100) for p in percentiles]
for i, (p, value) in enumerate(zip(percentiles, values)):
    if i == 0:
        print(f"{p}th percentile: {value:.3f} FFs")
    else:
        diff = value - values[i-1]
        print(f"{p}th percentile: {value:.3f} FFs (+{diff:.3f})")


###################################

# print("\nPLAYERS WITH 11 OR MORE SACKS (CAREER HIGH SEASON):")
# print("-" * 40)
elite_sacks = interior_dl[interior_dl['nfl_sacks_career_high'] >= 11]
# print(elite_sacks[['Name', 'nfl_sacks_career_high', 'nfl_pressures_career_high', 'career_nfl_sacks']].to_string(index=False))
#print(f"\nTotal players with 11+ sacks: {len(elite_sacks)}")


# print("\nPLAYERS WITH 36 OR MORE PRESSURES (CAREER HIGH SEASON):")
# print("-" * 40)
elite_pressures = interior_dl[interior_dl['nfl_pressures_career_high'] >= 36]
# print(elite_pressures[['Name', 'nfl_sacks_career_high', 'nfl_pressures_career_high', 'career_nfl_sacks']].to_string(index=False))
#print(f"\nTotal players with 36+ pressures: {len(elite_pressures)}")

# print("\nPLAYERS WITH 36 OR MORE PRESSURES (CAREER HIGH SEASON):")
# print("-" * 40)
elite_QBKDs = interior_dl[interior_dl['nfl_QBKDs_career_high'] >= 14]
# print(elite_QBKDs[['Name', 'nfl_sacks_career_high', 'nfl_pressures_career_high', 'nfl_QBKDs_career_high']].to_string(index=False))
#print(f"\nTotal players with 14+ QBKDs: {len(elite_QBKDs)}")



INTERIOR SUMMARY STATISTICS

PER GAME STATS

SACKS PER GAME PERCENTILES:
----------------------------------------
25th percentile: 0.063 sacks
50th percentile: 0.118 sacks (+0.055)
75th percentile: 0.200 sacks (+0.082)
90th percentile: 0.353 sacks (+0.153)
99th percentile: 0.643 sacks (+0.290)

PRESSURES PER GAME PERCENTILES:
----------------------------------------
25th percentile: 0.321 pressures
50th percentile: 0.458 pressures (+0.137)
75th percentile: 0.721 pressures (+0.264)
90th percentile: 1.152 pressures (+0.431)
99th percentile: 2.073 pressures (+0.921)

QBKDs PER GAME PERCENTILES:
----------------------------------------
25th percentile: 0.110 QBKDs
50th percentile: 0.167 QBKDs (+0.058)
75th percentile: 0.274 QBKDs (+0.107)
90th percentile: 0.447 QBKDs (+0.173)
99th percentile: 0.863 QBKDs (+0.416)

TFLs PER GAME PERCENTILES:
----------------------------------------
25th percentile: 0.189 TFLs
50th percentile: 0.262 TFLs (+0.073)
75th percentile: 0.402 TFLs (+0.140)
90th pe

***
### INTERIOR DL Summary Stats Takeaways

**1. TFLs per game is not as big of an indicator as I would've suspected.** The separation in tiers of TFLs per game is much smaller than I would've expected. But one not that we can take away from it is that the 75th percentile TFLs is .14 higher than 50th percentile, while the 50th percentile is only about half as much greater than the 25th percentile (+.073). This suggests that TFLs among great players is a noticibly higher clip than just average players. 

**2. SACKS MEAN ALMOST NOTHING.** Interior players simply don't get sacks like edge players do. Even in the 99th percentile of career HIGH sacks, we are just shy of 13 sacks in a season. A season high like that would only put an edge rusher somewhere between the 75th and 90th percentile. This means that we can't take much out of how many sacks an interior defensive linemen does or doesn't get. 

**3. Pressures on the other hand are a true separator.** For interior DL, pressures are the true separator. The gap between a 90th percentile pass rusher and a 99th percentile one (almost 1 full pressure per game) is massive. Elite interior pass rushers are rare because the jump from 'good' to 'elite' is enormous. 

**The bottom line:** 

***